## Inverse Kinemaics - Jacobian based methods 
- Test IK methods based on Jacobian
    - IK can be solved by multiplying Inverse Matrix of Jacobian
    - Several methods to solve inverse jacobian without singularity

Create UR environment

In [1]:
import mujoco
import mujoco_viewer # new viewer
import numpy as np
import time

In [2]:
model_path = "../ur5e_mjcf/scene.xml"

# declare model & data
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

Get Mujoco Jacobian

In [3]:
def get_jac_body_name(body_name=None):

    # initialize positional & rotational jacobian
    Jacobian_p = np.zeros((3,model.nu))
    Jacobian_r = np.zeros((3,model.nu))

    # get jacobian of end-effector
    mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(body_name).id)
    return Jacobian_p, Jacobian_r

### Function: calculate inverse jacobian

In [4]:
def get_invjac_name(body_name, method='svd', sigma_threshold=0.001, damping=1.0):

    jacobian_p, jacobian_r = get_jac_body_name(body_name)

    if method=='svd':
        # get inverse jacobian with Singular Value Decomposition
        U, Sigma, V_T = np.linalg.svd(jacobian_p, compute_uv=True)

        # past implementation - not good
        # Sigma_clipped_rev = np.minimum(1 / Sigma, upper_bound)

        # suppress singularities modifying sigma
        Sigma_clipped_rev = np.zeros_like(Sigma)
        for i, value in enumerate(Sigma):
            if Sigma[i] < sigma_threshold:
                Sigma_clipped_rev[i] = 0
            else:
                Sigma_clipped_rev[i] = 1/Sigma[i]

        # inverse matrix for position jacobian
        S_rev_matrix = np.zeros((model.nu,3)) # positional dimension = 3, dof = model.nu
        for i, value in enumerate(Sigma_clipped_rev):
            S_rev_matrix[i,i] = value
        J_inverse = V_T.T @ S_rev_matrix @ U.T

    if method=='DLS':
        # apply damped least squares
        pass
    
    return J_inverse

In [5]:
"""
Parameter tuning
- sigma threshold should not be big
    - if too big, robot will not move if it should move large to reach
    - small sigma = small gain = move large to get to goal
"""

'\nParameter tuning\n- sigma threshold should not be big\n    - if too big, robot will not move if it should move large to reach\n    - small sigma = small gain = move large to get to goal\n'

### MAIN loop: calculate error & update with forward

In [6]:
""" MAIN LOOP """

# create python viewer object
viewer = mujoco_viewer.MujocoViewer(model, data)

# initialize robot
init_qpos = [3.14, -0.8, -2.5, -2.2, 0.0, 0.0]

mujoco.mj_resetData(model, data)
data.qpos = init_qpos
mujoco.mj_forward(model, data) # first forward to get jacobian with no error

# goal position
goal_pos = [0.4, 0.4, 0.2]
body_name = "wrist_3_link"

# scale down error
alpha = 0.02

while True:
    if viewer.is_alive:

        # get inverse jacobian & unit error vector
        J_inverse = get_invjac_name(body_name=body_name, method='svd')
        error = goal_pos - data.body(body_name).xpos.copy()
        print(f"current error: {error}")

        dq = alpha * (J_inverse @ error)
        print(f"dq: {dq}")
        data.qpos += dq
        # print(f"qpos before update: {qpos_before} \n qpos after update: {data.qpos}")

        mujoco.mj_forward(model, data)

        print(f"current body pos: {data.body(body_name).xpos}")

        # terminalize
        if np.linalg.norm(goal_pos - data.body(body_name).xpos) < 0.02:
            print("IK done.")
            break

        viewer.render()

    else:
        break

# close
viewer.close()

current error: [ 0.26574288  0.23866787 -0.13517305]
dq: [-3.28529138e-02 -4.28409617e-02  7.42561304e-03  1.16082873e-02
  2.87236694e-18  0.00000000e+00]
current body pos: [0.13980043 0.16601937 0.33224785]
current error: [ 0.26019957  0.23398063 -0.13224785]
dq: [-0.02951786 -0.04058567  0.00779773  0.01086656  0.          0.        ]
current body pos: [0.145203   0.17062106 0.32939886]
current error: [ 0.254797    0.22937894 -0.12939886]
dq: [-2.66899943e-02 -3.85253007e-02  8.11034029e-03  1.02030153e-02
  7.98250423e-19  0.00000000e+00]
current body pos: [0.15047299 0.1751379  0.32662231]
current error: [ 0.24952701  0.2248621  -0.12662231]
dq: [-2.42655421e-02 -3.66286064e-02  8.37095699e-03  9.60547420e-03
 -1.29513870e-17  0.00000000e+00]
current body pos: [0.15561712 0.17957068 0.32391501]
current error: [ 0.24438288  0.22042932 -0.12391501]
dq: [-2.21673409e-02 -3.48729883e-02  8.58594162e-03  9.06464237e-03
 -8.10470654e-18  0.00000000e+00]
current body pos: [0.16064113 0.1